# 28 - Arboles de decision: Gini, entropia y fronteras de decision

Un arbol de decision aprende a hacer preguntas simples sobre los datos: por ejemplo, `petal length <= 2.45` o `ingreso <= 12000`.
Cada pregunta divide el conjunto de datos en grupos mas pequenos. Al final, cada hoja contiene una prediccion.

La meta de este notebook es entender el modelo por dentro y practicarlo con visualizaciones, no solo llamar a `sklearn`.

Al terminar deberias poder:

1. Explicar que son raiz, nodo, rama, hoja, profundidad y criterio de particion.
2. Calcular e interpretar impureza de Gini y entropia.
3. Ver como un arbol parte el espacio de variables en regiones rectangulares.
4. Entrenar, visualizar y regularizar arboles de decision con `scikit-learn`.
5. Detectar sobreajuste y justificar decisiones de modelado.
6. Aplicar arboles a clasificacion y regresion.


## Ruta de la sesion

1. Intuicion: un arbol como secuencia de preguntas.
2. Impureza: Gini y entropia.
3. Como se elige una particion.
4. Visualizacion de fronteras de decision.
5. Sobreajuste y regularizacion.
6. Interpretacion: reglas, arbol dibujado e importancia de variables.
7. Arboles de regresion.
8. Ejercicios para pensar.


## 1) Preparacion

Este notebook usa solo librerias comunes del curso: `numpy`, `pandas`, `matplotlib` y `scikit-learn`.
Los datos son autocontenidos: se generan o vienen incluidos en `scikit-learn`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 28

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 11


## 2) Intuicion: que aprende un arbol

Un arbol de decision no aprende una ecuacion suave como una recta. Aprende una lista de reglas del tipo:

- si `variable <= umbral`, ve a la izquierda;
- si no, ve a la derecha;
- repite hasta llegar a una hoja;
- la hoja predice la clase mas frecuente o el promedio de los valores.

Vocabulario basico:

| Concepto | Significado |
|---|---|
| Raiz | Primer nodo del arbol; contiene todos los datos de entrenamiento. |
| Nodo interno | Punto donde se hace una pregunta. |
| Rama | Camino que sale de una pregunta. |
| Hoja | Nodo final que produce la prediccion. |
| Profundidad | Numero de preguntas desde la raiz hasta una hoja. |
| Particion | Division de datos producida por una pregunta. |
| Impureza | Medida de mezcla dentro de un nodo. Baja impureza significa grupos mas homogeneos. |

Idea central: un buen corte deja a cada lado grupos mas puros que el grupo original.


## 3) Impureza: por que Gini y entropia existen

Imagina un nodo con ejemplos de varias clases. Si todos son de la misma clase, el nodo esta puro.
Si las clases estan mezcladas, el nodo esta impuro.

Un arbol necesita una forma numerica de responder:

> Que tan mezclado esta este nodo?

Dos respuestas clasicas son el indice de Gini y la entropia.


### 3.1 Indice de Gini

Para un nodo con proporciones de clase `p_1, p_2, ..., p_K`:

$$
Gini = 1 - \sum_{k=1}^{K} p_k^2
$$

Intuicion:

- `Gini = 0`: nodo puro.
- Gini crece cuando las clases estan mas mezcladas.
- En clasificacion binaria, el maximo ocurre cerca de 50% y 50%.

Otra forma de pensarlo: Gini mide que tan probable seria equivocarte si asignaras una etiqueta al azar siguiendo las proporciones del nodo.


### 3.2 Entropia

Para el mismo nodo:

$$
Entropia = -\sum_{k=1}^{K} p_k \log_2(p_k)
$$

Intuicion:

- `Entropia = 0`: no hay incertidumbre; todos los ejemplos son de la misma clase.
- Entropia alta: hay mas incertidumbre sobre la clase.
- En clasificacion binaria, tambien alcanza su maximo cerca de 50% y 50%.

La entropia viene de teoria de informacion. Aqui la usamos como medida de incertidumbre antes de predecir.


In [ ]:
def gini_from_counts(counts):
    """Calcula Gini a partir de conteos por clase."""
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    return 1 - np.sum(p ** 2)


def entropy_from_counts(counts):
    """Calcula entropia base 2 a partir de conteos por clase."""
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total == 0:
        return 0.0
    p = counts / total
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


examples = {
    "puro: [20, 0]": [20, 0],
    "casi puro: [18, 2]": [18, 2],
    "mezclado: [10, 10]": [10, 10],
    "tres clases balanceadas: [10, 10, 10]": [10, 10, 10],
    "tres clases desbalanceadas: [25, 4, 1]": [25, 4, 1],
}

rows = []
for name, counts in examples.items():
    total = sum(counts)
    proportions = [round(c / total, 3) for c in counts]
    rows.append({
        "caso": name,
        "proporciones": proportions,
        "gini": round(gini_from_counts(counts), 4),
        "entropia": round(entropy_from_counts(counts), 4),
    })

pd.DataFrame(rows)


### 3.3 Curvas de Gini y entropia en dos clases

En dos clases basta pensar en una proporcion `p` para la clase 1 y `1 - p` para la clase 0.
El nodo es mas impuro cuando `p` esta cerca de `0.5`.


In [ ]:
p = np.linspace(0, 1, 401)
gini_curve = 1 - (p ** 2 + (1 - p) ** 2)

entropy_curve = np.zeros_like(p)
mask = (p > 0) & (p < 1)
entropy_curve[mask] = -p[mask] * np.log2(p[mask]) - (1 - p[mask]) * np.log2(1 - p[mask])

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(p, gini_curve, label="Gini")
ax.plot(p, entropy_curve, label="Entropia")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_title("Impureza para un nodo binario")
ax.set_xlabel("Proporcion de la clase 1")
ax.set_ylabel("Impureza")
ax.legend()
plt.show()


Observa que ambas curvas cuentan una historia parecida: pureza en los extremos, maxima mezcla cerca del centro.
En la practica, `gini` y `entropy` suelen producir arboles similares, aunque no siempre identicos.


## 4) Como decide un arbol donde cortar

Para cada variable y para muchos umbrales posibles, el algoritmo prueba una pregunta como:

```text
variable <= umbral
```

Despues calcula la impureza ponderada de los dos hijos:

$$
Impureza\ ponderada = \frac{n_{izq}}{n} I(izq) + \frac{n_{der}}{n} I(der)
$$

Elige el corte con menor impureza ponderada, o equivalentemente, el que produce mayor reduccion de impureza.

Importante: el algoritmo usual en arboles tipo CART es **codicioso**. Elige el mejor corte ahora, no necesariamente el mejor arbol global posible.


## 5) Datos: Iris con dos variables

Usaremos el dataset Iris porque es pequeno, visual y viene incluido en `scikit-learn`.
Para poder dibujar fronteras de decision en 2D, usaremos solo dos variables:

- `petal length (cm)`
- `petal width (cm)`


In [ ]:
iris = load_iris(as_frame=True)
df_iris = iris.frame.copy()
df_iris["species"] = df_iris["target"].map(dict(enumerate(iris.target_names)))

features_2d = ["petal length (cm)", "petal width (cm)"]
X_iris = df_iris[features_2d].to_numpy()
y_iris = df_iris["target"].to_numpy()

print(df_iris.shape)
df_iris.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for class_id, class_name in enumerate(iris.target_names):
    mask = y_iris == class_id
    ax.scatter(
        X_iris[mask, 0],
        X_iris[mask, 1],
        label=class_name,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.85,
    )

ax.set_title("Iris: dos variables para visualizar el problema")
ax.set_xlabel(features_2d[0])
ax.set_ylabel(features_2d[1])
ax.legend(title="species")
plt.show()


## 6) Buscar un corte a mano

Vamos a simular una parte del algoritmo. Para una variable, probaremos muchos umbrales y mediremos la impureza ponderada.
El mejor corte sera el que deje menor impureza despues de dividir.


In [ ]:
def weighted_impurity_for_threshold(x, y, threshold, criterion="gini"):
    """Impureza ponderada al partir una sola variable con x <= threshold."""
    left = x <= threshold
    right = ~left

    if left.sum() == 0 or right.sum() == 0:
        return np.nan

    n_classes = int(np.max(y)) + 1
    impurity_fn = gini_from_counts if criterion == "gini" else entropy_from_counts

    left_counts = np.bincount(y[left], minlength=n_classes)
    right_counts = np.bincount(y[right], minlength=n_classes)

    n = len(y)
    left_weight = left.sum() / n
    right_weight = right.sum() / n

    return left_weight * impurity_fn(left_counts) + right_weight * impurity_fn(right_counts)


def candidate_thresholds(x):
    values = np.sort(np.unique(x))
    return (values[:-1] + values[1:]) / 2


In [ ]:
criterion = "gini"
rows = []

for feature_idx, feature_name in enumerate(features_2d):
    x = X_iris[:, feature_idx]
    thresholds = candidate_thresholds(x)
    scores = np.array([
        weighted_impurity_for_threshold(x, y_iris, t, criterion=criterion)
        for t in thresholds
    ])
    best_idx = np.nanargmin(scores)
    rows.append({
        "variable": feature_name,
        "mejor_umbral": thresholds[best_idx],
        "impureza_ponderada": scores[best_idx],
    })

pd.DataFrame(rows).sort_values("impureza_ponderada")


In [ ]:
feature_idx = 0
feature_name = features_2d[feature_idx]
x = X_iris[:, feature_idx]
thresholds = candidate_thresholds(x)
scores = np.array([
    weighted_impurity_for_threshold(x, y_iris, t, criterion="gini")
    for t in thresholds
])
best_idx = np.nanargmin(scores)
best_threshold = thresholds[best_idx]

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(thresholds, scores, marker="o", markersize=3, linewidth=1)
ax.axvline(best_threshold, color="red", linestyle="--", label=f"mejor umbral = {best_threshold:.2f}")
ax.set_title(f"Busqueda de corte para {feature_name}")
ax.set_xlabel("Umbral")
ax.set_ylabel("Impureza ponderada despues del corte")
ax.legend()
plt.show()


La grafica anterior muestra una idea clave: el arbol no entiende flores ni botanica. Solo prueba cortes y escoge los que reducen mejor la mezcla de clases.


## 7) Entrenar un arbol pequeno

Un arbol con `max_depth=1` se llama a veces *stump* o tocon: solo puede hacer una pregunta.
Esto permite comparar nuestra busqueda manual con lo que aprende `scikit-learn`.


In [ ]:
stump = DecisionTreeClassifier(max_depth=1, criterion="gini", random_state=RANDOM_STATE)
stump.fit(X_iris, y_iris)

print(export_text(stump, feature_names=features_2d))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
plot_tree(
    stump,
    feature_names=features_2d,
    class_names=list(iris.target_names),
    filled=True,
    rounded=True,
    impurity=True,
    proportion=True,
    ax=ax,
)
ax.set_title("Arbol de profundidad 1")
plt.show()


## 8) Ver como el arbol parte el espacio

En dos dimensiones, cada pregunta del tipo `x_j <= umbral` crea una linea vertical u horizontal.
Con varias preguntas, el arbol construye regiones rectangulares.


In [ ]:
def plot_decision_regions(model, X, y, ax=None, feature_names=None, class_names=None, title=""):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    x_min, x_max = X[:, 0].min() - 0.6, X[:, 0].max() + 0.6
    y_min, y_max = X[:, 1].min() - 0.4, X[:, 1].max() + 0.4
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 450),
        np.linspace(y_min, y_max, 450),
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.22, cmap="Set2")
    ax.contour(xx, yy, Z, colors="black", linewidths=0.45, alpha=0.35)

    for class_id in np.unique(y):
        mask = y == class_id
        label = class_names[class_id] if class_names is not None else str(class_id)
        ax.scatter(
            X[mask, 0],
            X[mask, 1],
            label=label,
            edgecolor="black",
            linewidth=0.4,
            s=45,
            alpha=0.9,
        )

    if feature_names is not None:
        ax.set_xlabel(feature_names[0])
        ax.set_ylabel(feature_names[1])
    ax.set_title(title)
    ax.legend(loc="best", fontsize=9)
    return ax


In [ ]:
depths = [1, 2, 3, None]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

for ax, depth in zip(axes.ravel(), depths):
    model = DecisionTreeClassifier(max_depth=depth, criterion="gini", random_state=RANDOM_STATE)
    model.fit(X_iris, y_iris)
    train_acc = model.score(X_iris, y_iris)
    depth_label = "sin limite" if depth is None else str(depth)
    plot_decision_regions(
        model,
        X_iris,
        y_iris,
        ax=ax,
        feature_names=features_2d,
        class_names=list(iris.target_names),
        title=f"max_depth={depth_label} | accuracy train={train_acc:.2f}",
    )

plt.show()


Observa el patron:

- profundidad baja: fronteras simples, interpretables, pero pueden quedarse cortas;
- profundidad alta: fronteras mas detalladas, pero pueden aprender ruido;
- sin limite: el arbol puede crear reglas muy especificas para ejemplos particulares.


## 9) Gini contra entropia

`scikit-learn` permite elegir el criterio con `criterion="gini"`, `criterion="entropy"` o `criterion="log_loss"`.
Aqui compararemos Gini y entropia en el mismo problema.


In [ ]:
criteria = ["gini", "entropy"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for ax, criterion in zip(axes, criteria):
    model = DecisionTreeClassifier(max_depth=3, criterion=criterion, random_state=RANDOM_STATE)
    model.fit(X_iris, y_iris)
    plot_decision_regions(
        model,
        X_iris,
        y_iris,
        ax=ax,
        feature_names=features_2d,
        class_names=list(iris.target_names),
        title=f"criterion={criterion} | accuracy train={model.score(X_iris, y_iris):.2f}",
    )

plt.show()


No busques una regla universal como "entropia siempre es mejor". La eleccion depende de datos, ruido, validacion y objetivo.
En muchos casos, la diferencia practica es pequena frente a decisiones como profundidad maxima o tamano minimo de hoja.


## 10) Sobreajuste: cuando el arbol memoriza

Los arboles son modelos de alta varianza: si los dejamos crecer sin restricciones, pueden ajustar detalles accidentales de los datos.
Vamos a usar datos sinteticos con ruido para ver la diferencia entre desempeno en entrenamiento y prueba.


In [ ]:
X_moons, y_moons = make_moons(n_samples=420, noise=0.28, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons,
    y_moons,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_moons,
)

fig, ax = plt.subplots(figsize=(7, 5))
for class_id in np.unique(y_moons):
    mask = y_moons == class_id
    ax.scatter(
        X_moons[mask, 0],
        X_moons[mask, 1],
        label=f"clase {class_id}",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.8,
    )
ax.set_title("Datos sinteticos con ruido")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend()
plt.show()


In [ ]:
rows = []
for depth in range(1, 16):
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    rows.append({
        "max_depth": depth,
        "accuracy_train": accuracy_score(y_train, model.predict(X_train)),
        "accuracy_test": accuracy_score(y_test, model.predict(X_test)),
        "n_leaves": model.get_n_leaves(),
    })

scores_depth = pd.DataFrame(rows)
scores_depth


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(scores_depth["max_depth"], scores_depth["accuracy_train"], marker="o", label="train")
ax1.plot(scores_depth["max_depth"], scores_depth["accuracy_test"], marker="o", label="test")
ax1.set_xlabel("max_depth")
ax1.set_ylabel("accuracy")
ax1.set_ylim(0.65, 1.02)
ax1.legend(loc="lower right")
ax1.set_title("Profundidad, desempeno y sobreajuste")

ax2 = ax1.twinx()
ax2.plot(scores_depth["max_depth"], scores_depth["n_leaves"], color="gray", linestyle="--", marker="s", label="hojas")
ax2.set_ylabel("numero de hojas")
ax2.grid(False)
plt.show()


Interpretacion esperada:

- El accuracy de entrenamiento suele subir al aumentar la profundidad.
- El accuracy de prueba puede mejorar al inicio y luego estancarse o bajar.
- Mas hojas no siempre significan mejor generalizacion.


In [ ]:
depths = [1, 3, 6, None]
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

for ax, depth in zip(axes.ravel(), depths):
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    depth_label = "sin limite" if depth is None else str(depth)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    plot_decision_regions(
        model,
        X_moons,
        y_moons,
        ax=ax,
        feature_names=["x1", "x2"],
        class_names=["0", "1"],
        title=f"max_depth={depth_label} | accuracy test={test_acc:.2f}",
    )

plt.show()


### Parametros utiles para regularizar

| Parametro | Que controla | Efecto comun |
|---|---|---|
| `max_depth` | Profundidad maxima | Reduce complejidad si se baja. |
| `min_samples_split` | Minimo de ejemplos para partir un nodo | Evita cortes con pocos datos. |
| `min_samples_leaf` | Minimo de ejemplos en cada hoja | Suaviza fronteras y reduce hojas pequenas. |
| `max_leaf_nodes` | Numero maximo de hojas | Limita cantidad de reglas finales. |
| `criterion` | Medida de impureza | Cambia como se evalua cada corte. |
| `class_weight` | Peso de clases | Ayuda cuando hay clases desbalanceadas. |
| `ccp_alpha` | Poda por complejidad de costo | Elimina ramas que aportan poco. |

En proyectos reales, estos parametros se eligen con validacion, no viendo el conjunto de prueba una y otra vez.


## 11) Evaluar un arbol regularizado

Ahora entrenamos un arbol con restricciones razonables y miramos matriz de confusion y reporte de clasificacion.


In [ ]:
regularized_tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=8,
    random_state=RANDOM_STATE,
)
regularized_tree.fit(X_train, y_train)

y_pred = regularized_tree.predict(X_test)
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_estimator(regularized_tree, X_test, y_test, ax=ax, cmap="Blues")
ax.set_title("Matriz de confusion en prueba")
plt.show()


Una matriz de confusion ayuda a salir del numero unico de accuracy. Preguntas utiles:

- Que clase se confunde mas?
- El error tiene el mismo costo en ambas clases?
- Necesitamos ajustar el modelo, el umbral de decision o la forma de recolectar datos?


## 12) Interpretacion: reglas e importancia de variables

Los arboles son populares porque pueden traducirse a reglas.
Pero interpretabilidad no significa automaticamente verdad causal.
Un arbol puede descubrir patrones del conjunto de entrenamiento que no representan mecanismos reales.


In [ ]:
interpretable_tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
)
interpretable_tree.fit(X_iris, y_iris)

print(export_text(interpretable_tree, feature_names=features_2d))


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
plot_tree(
    interpretable_tree,
    feature_names=features_2d,
    class_names=list(iris.target_names),
    filled=True,
    rounded=True,
    impurity=True,
    proportion=True,
    ax=ax,
)
ax.set_title("Arbol interpretable para Iris")
plt.show()


In [ ]:
X_iris_all = df_iris[iris.feature_names]
y_iris_all = df_iris["target"]

full_feature_tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=4,
    random_state=RANDOM_STATE,
)
full_feature_tree.fit(X_iris_all, y_iris_all)

importances = pd.Series(
    full_feature_tree.feature_importances_,
    index=iris.feature_names,
).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.8))
importances.plot(kind="barh", ax=ax, color="#3b7a9e")
ax.set_title("Importancia de variables segun el arbol")
ax.set_xlabel("Importancia")
plt.show()


Cuidado con la importancia de variables:

- Es importancia dentro de este modelo, no una ley general.
- Puede cambiar con otra muestra de datos.
- No prueba causalidad.
- Puede favorecer variables con muchos posibles cortes.


## 13) Arboles de regresion

Un arbol tambien puede predecir valores numericos.
La idea es parecida, pero en vez de buscar hojas puras por clase, busca hojas donde los valores de `y` sean parecidos.

En `DecisionTreeRegressor`, la prediccion de una hoja suele ser el promedio de los valores de entrenamiento dentro de esa hoja.
El resultado se ve como una funcion por escalones.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X_reg = np.linspace(0, 10, 220).reshape(-1, 1)
y_reg = (
    np.sin(X_reg.ravel())
    + 0.35 * np.sin(3 * X_reg.ravel())
    + rng.normal(0, 0.18, size=X_reg.shape[0])
)

x_grid = np.linspace(0, 10, 700).reshape(-1, 1)
models = {
    "max_depth=2": DecisionTreeRegressor(max_depth=2, random_state=RANDOM_STATE),
    "max_depth=5": DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE),
    "sin limite": DecisionTreeRegressor(random_state=RANDOM_STATE),
}

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.scatter(X_reg.ravel(), y_reg, s=20, alpha=0.55, edgecolor="black", linewidth=0.2, label="datos")

for label, model in models.items():
    model.fit(X_reg, y_reg)
    ax.plot(x_grid.ravel(), model.predict(x_grid), linewidth=2, label=label)

ax.set_title("Arboles de regresion: predicciones por escalones")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()


Observa que el arbol de regresion no interpola suavemente. Divide el eje `x` en intervalos y predice un promedio por intervalo.
Esto puede ser muy util para relaciones no lineales, pero tambien puede sobreajustar ruido si crece demasiado.


## 14) Ventajas, limites y usos razonables

Ventajas:

- Faciles de visualizar cuando son pequenos.
- Requieren poco preprocesamiento.
- Capturan interacciones no lineales.
- Funcionan con variables numericas y categoricas si estas se codifican adecuadamente.

Limites:

- Pueden sobreajustar con facilidad.
- Son inestables: pequenos cambios en datos pueden cambiar el arbol.
- Las fronteras son rectangulares, lo cual no siempre representa bien el fenomeno.
- Un arbol profundo pierde interpretabilidad.

Usos razonables:

- Modelo base interpretable.
- Exploracion inicial de reglas.
- Problemas donde las reglas tipo umbral son naturales.
- Punto de partida para entender modelos de ensamble como Random Forest y Gradient Boosting.


## 15) Checklist de aprendizaje

Antes de pasar al siguiente tema, intenta responder sin ver la teoria:

1. Que significa que un nodo sea puro?
2. Por que Gini y entropia son cero en un nodo puro?
3. Que calcula la impureza ponderada despues de un corte?
4. Por que un arbol profundo puede tener excelente accuracy de entrenamiento y peor accuracy de prueba?
5. Que parametros usarias para hacer un arbol menos complejo?
6. Por que la importancia de variables no debe interpretarse como causalidad?


## 16) Ejercicios para pensar y practicar

### Ejercicio 1: calcular impureza a mano

Tienes tres nodos con conteos por clase:

- A: `[30, 0]`
- B: `[15, 15]`
- C: `[24, 6]`

1. Calcula Gini para cada nodo.
2. Calcula entropia para cada nodo.
3. Ordenalos de mas puro a mas impuro.
4. Explica con palabras por que ese orden tiene sentido.

### Ejercicio 2: elegir entre dos cortes

Un nodo tiene 100 ejemplos. Se prueban dos cortes:

- Corte 1: izquierda `[45, 5]`, derecha `[10, 40]`.
- Corte 2: izquierda `[30, 20]`, derecha `[25, 25]`.

1. Calcula la impureza Gini ponderada de cada corte.
2. Cual elegiria el arbol si usa Gini?
3. Que pasaria si el costo de equivocarse en la clase 1 fuera mucho mayor?

### Ejercicio 3: experimentar sin copiar

Con el dataset `make_moons` de este notebook:

1. Entrena arboles con `max_depth` de 1 a 12.
2. Repite el experimento usando `min_samples_leaf` igual a 1, 5, 10 y 20.
3. Construye una tabla con accuracy de train y test.
4. Elige un modelo y justifica tu eleccion en 5 lineas maximo.

### Ejercicio 4: comparar Gini y entropia

Usa Iris con las cuatro variables.

1. Entrena dos arboles: uno con `criterion="gini"` y otro con `criterion="entropy"`.
2. Usa la misma profundidad maxima.
3. Compara reglas con `export_text`.
4. Identifica una regla que sea igual y una que cambie.
5. Explica si el cambio importa para el problema.

### Ejercicio 5: frontera de decision

Crea una nueva grafica con fronteras de decision para `max_depth=1`, `2`, `4` y `8`.

1. Que frontera parece subajustar?
2. Que frontera parece sobreajustar?
3. Cual explicarias mejor a una persona no tecnica?
4. Cual elegirias si el objetivo fuera predecir bien datos nuevos?

### Ejercicio 6: mini implementacion

Implementa una funcion `best_split_one_feature(x, y, criterion)` que regrese:

- mejor umbral;
- impureza ponderada;
- conteos por clase a la izquierda;
- conteos por clase a la derecha.

No uses `DecisionTreeClassifier` dentro de la funcion. La meta es entender el algoritmo.

### Ejercicio 7: arboles y responsabilidad

Piensa en un arbol que decide si una persona recibe una beca, credito o atencion prioritaria.

1. Que variables podrian introducir sesgo?
2. Que errores serian mas costosos?
3. Que metricas revisarias ademas de accuracy?
4. Que explicacion minima deberia recibir una persona afectada por la decision?


## 17) Cierre

Un arbol de decision es potente porque convierte datos en reglas. Su fortaleza tambien es su riesgo: si lo dejas crecer sin control, puede convertir ruido en reglas aparentemente convincentes.

La practica responsable consiste en mirar tres cosas al mismo tiempo:

1. desempeno en datos no vistos;
2. complejidad del arbol;
3. sentido del problema real que se quiere resolver.
